In [0]:
import importlib
import src.pipeline as pipeline
import pandas as pd

importlib.reload(pipeline)

executar_pipeline = pipeline.executar_pipeline

In [0]:
resultados = executar_pipeline(
    spark,
    "workspace.default.manufacturing_defect_dataset"
)

In [0]:
display(resultados["tabela_resumo"])

## 1. Carregamento dos dados
Nesta etapa, carregamos o dataset original armazenado no Databricks. O objetivo é estabelecer a base inicial da análise e verificar se a tabela está disponível para processamento.

In [0]:
from pyspark.sql import functions as F

# carrega a tabela original do Databricks
df = spark.read.table("workspace.default.manufacturing_defect_dataset")

# exibe a estrutura da tabela
df.printSchema() 

# contagem de registros
print(f"Total de registros: {df.count()}")

# exibe os primeiros registros
display(df.limit(20))

## 2. Inspeção inicial
### Auditoria de qualidade dos dados
Nesta etapa, avaliamos a integridade da estrutura da base, verificando valores nulos, duplicados e tipos de dados.
A auditoria tem o objetivo de identificar problemas que possam comprometer as análises posteriores.

In [0]:
# valores nulos por coluna
nulos = df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

display(nulos)

# total de registros
total_linhas = df.count()

# total de registros distintos
total_distintos = df.distinct().count()

# duplicados
duplicadas = total_linhas - total_distintos

print(f"Total de registros: {total_linhas}")
print(f"Total de registros distintos: {total_distintos}")
print(f"Total de duplicatas: {duplicadas}")


## Validando Regras de Negócio
Validação se os valores registrados fazem sentido dentro das regras que conhecemos sobre o processo, pois além de verificar se um número é negativo precisamos entender o significado da variável. 

Exemplo:
- Volume de produção negativo não faz sentido.
- Custo de produção negativo não faz sentido.
- Downtime percentual acima de 100% não faz sentido se representa percentual do tempo.
- Uma taxa de defeitos acima de 100% seria suspeita.

In [0]:

# verifica regras de negócio
regras_invalidas = df.filter(
    (F.col('MaintenanceHours') < 0) |
    (F.col('ProductionVolume') <= 0) |
    (F.col('ProductionCost') < 0) | 
    (F.col('DowntimePercentage') < 0) | 
    (F.col('DowntimePercentage') > 100) | 
    (F.col('EnergyEfficiency') <= 0) 
    )

print(f"Quantidade de registros inválidos: {regras_invalidas.count()}")
display(regras_invalidas)

## 3. Preparação dos dados
Nesta etapa, serão criados indicadores para apoiar as análises de qualidade, custos e eficiência operacional.
Os indicadores serão calculados a partir das variáveis originais, sem remover registros apenas por apresentarem valores extremos.

Após ter uma auditoria inicial, iremos criar a camada Silver e o próximo passo é a base analítica:
- Preservar todos os registros válidos
- Não remover outliers automaticamente
- Criar indicadores para a análise

In [0]:
# cria a base analítica inicial
df_silver = df 

# cria coluna de custo por unidade produzida
df_silver = df_silver.withColumn(
    "UnitCost",
    F.when(
        F.col("ProductionVolume") > 0,
        F.col("ProductionCost") / F.col("ProductionVolume")
    )
)

# cria coluna de consumo energético por unidade produzida
df_silver = df_silver.withColumn(
    "EnergyPerUnit",
    F.when(
        F.col("ProductionVolume") > 0,
        F.col("EnergyConsumption") / F.col("ProductionVolume")
    )
)

# exibe as colunas criadas
display(
    df_silver.select(
        "ProductionVolume",
        "ProductionCost",
        "UnitCost",
        "EnergyConsumption",
        "EnergyPerUnit"
    ).limit(10)
)

### Explicação para a criação dessas colunas
- UnitCost: permite comparar custos relativos ao volume de produção.
- EnergyPerUnit: permite comparar consumo energético relativo ao volume produzido. 

Essas colunas ajudam porque comparar custos totais ou o consumo total entre registros de volumes diferentes pode gerar interpretações equivocadas. 
Exemplo: uma operação que produz 1.000 unidades pode consumir mais energia que outra que produz 100 unidades, mas não significa necessariamente que é menos eficiente. Ter esse dado unitário ajuda a investigar essa diferença. 

## 4. Salvar a camada Silver 
Nesta etapa, iremos salvar os dados tratados como uma tabela Delta para poder reutilizar em outras análises. 
Isso serve para reutilizar os dados sem repetir todas as transformações, poder consultar a base com SQL, conectar o Power BI na camada analítica e também manter uma estrutura organizada no projeto.

In [0]:
# cria nome da tabela
tabela_silver = "workspace.default.manufacturing_defects_silver"

# salva a camada silver como tabela delta
(
    df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela_silver)
)
# "overwrite" substitui a tabela de destino caso ela já exista
print(f"Tabela salva: {tabela_silver}")

In [0]:
# faz a leitura novamente da tabela para conferir se foi salva
df_silver_confere = spark.read.table("workspace.default.manufacturing_defects_silver")

# confere a quantidade de registros
print(f"Registros na camada Silver: {df_silver_confere.count()}")

# confere a estrutura
df_silver_confere.printSchema()

# exibe os dados em formato de tabela
display(df_silver_confere.limit(15))

## 5. Detecção de outliers no custo unitário

Calcula o custo unitário e usa o intervalo interquartil (IQR) para identificar e filtrar outliers.

In [0]:
# Calcula os quartis do custo unitário
quartis = df_silver.agg(
    F.expr('percentile(UnitCost, 0.25)').alias('q1'),
    F.expr('percentile(UnitCost, 0.75)').alias('q3')
).collect()[0]
q1 = quartis['q1']
q3 = quartis['q3']
iqr = q3 - q1

# Define os limites inferior e superior para outliers
limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr

print(f"Q1: {q1:.2f}")
print(f"Q3: {q3:.2f}")
print(f"IQR: {iqr:.2f}")

print(f"Limite inferior: {limite_inferior:.2f}")
print(f"Limite superior: {limite_superior:.2f}")

df_silver = df_silver.withColumn(
    "IsUnitCostOutlier",
    (F.col("UnitCost") < limite_inferior) |
    (F.col("UnitCost") > limite_superior)
)

df_silver.groupBy("IsUnitCostOutlier").count().show()

Como o limite inferior ficou um valor negativo, significa que nenhum custo unitário positivo seria considerado outlier pela regra do inferior. Com isso, entende-se que os 284 registros que foram identificados como 'true',  estão acima do limite superior de 72.

Porém, ter registros que ultrapassam o limite de 72 não significa necessariamente que esteja errado. Pode ter acontecido do volume produzido ser baixo, o custo tenha sido elevado, ou ainda exista uma situação operacional para investigar.

Por isso, o próximo passo é analisar se os registros com custo unitário extremo apresentam indicadores diferentes de qualidade, manutenção e eficiência. 

In [0]:
df_silver.groupBy("IsUnitCostOutlier").agg(
    F.count("*").alias("TotalRegistros"),
    F.avg("UnitCost").alias("CustoUnitarioMedio"),
    F.avg("ProductionCost").alias("CustoProducaoMedio"),
    F.avg("QualityScore").alias("QualidadeMedia"),
    F.avg("DefectRate").alias("TaxaDefeitoMedia"),
    F.avg("MaintenanceHours").alias("HorasManutencaoMedia"),
    F.avg("EnergyPerUnit").alias("EnergiaPorUnidadeMedia"),
    F.avg("DefectStatus").alias("MediaDefectStatus")).show()

A análise dos outliers de custo unitário mostra uma associação entre custos unitários elevados e maior consumo de energia por unidade, enquanto que as médias de qualidade e taxa de defeitos apresentaram diferenças pequenas entre os grupos. 

Os resultados não permitem afirmar relações de causa e efeito. Ainda não serão excluídos esses 284 registros, pois eles podem representar situações operacionais relevantes, e não erros nos dados.

## 6. Análise exploratória dos dados

### Frente 1 — Qualidade e defeitos

Análise dos níveis de defeitos e sua relação com indicadores de qualidade e manutenção.

In [0]:
# verifica se existe equilibrio entre baixo e alto nível de defeitos
df_silver.groupBy("DefectStatus").agg(
    F.count("*").alias("TotalRegistros"),
    F.round(F.count("*") / df_silver.count() * 100, 2).alias("Percentual")
).orderBy("DefectStatus").show()

A análise da distribuição da coluna 'DefectStatus' identificou que existe um desequilíbrio entre as classificações, sendo que 84,04% dos registros apresentam alto nível de defeitos, enquanto que 15,96% apresentam baixo nível. Isso deve ser considerado na interpretação das análises.

In [0]:
df_silver.groupBy("DefectStatus").agg(
    F.count("*").alias("TotalRegistros"),
    F.round(F.avg("DefectRate"), 2).alias("TaxaDefeitoMedia"),
    F.round(F.avg("QualityScore"), 2).alias("QualidadeMedia"),
    F.round(F.avg("MaintenanceHours"), 2).alias("HorasManutencaoMedia"),
    F.round(F.avg("ProductionVolume"), 2).alias("VolumeProducaoMedio"),
    F.round(F.avg("UnitCost"), 2).alias("CustoUnitarioMedio"),
    F.round(F.avg("EnergyPerUnit"), 2).alias("EnergiaPorUnidadeMedia")
).orderBy("DefectStatus").show()

In [0]:
import matplotlib.pyplot as plt

# agrupa os dados pela classificação de defeitos
df_grafico = df_silver.groupBy("DefectStatus").agg(F.round(F.avg("DefectRate"), 2).alias("TaxaDefeitoMedia")).orderBy("DefectStatus")

# converte o resultado do PySpark para Pandas
df_grafico_pd = df_grafico.toPandas()

# exibe a tabela que será utilizada no gráfico
display(df_grafico_pd)

In [0]:
# variaveis para definir os dados do gráfico
categorias = ["Baixo nível de defeitos", "Alto nível de defeitos"]
valores = df_grafico_pd["TaxaDefeitoMedia"]

# cria o gráfico
plt.figure(figsize=(9, 5))
plt.bar(categorias, valores)

# personaliza o gráfico
plt.title("Registros com alto nível de defeitos apresentam maior taxa média")
plt.xlabel("Classificação de defeitos")
plt.ylabel("Taxa média de defeitos")

# exibe os valores acima das barras
for i, valor in enumerate(valores):
    plt.text(i, valor + 0.03, str(valor), ha="center")

plt.tight_layout()
plt.show()

### Interpretação do gráfico - Taxa média de defeitos

Demonstra que os registros classificados com alto nível de defeitos apresentam uma taxa média de defeitos de 2,89 enquanto os registros com baixo nível apresentam uma média de 2,01.

Essa diferença mostra uma variação entre os dois grupos, indicando que a classificação de alto nível de defeitos está associada a uma maior taxa média de defeitos. Esse resultado é coerente com a classificação utilizada no dataset e ajuda a caracterizar os diferentes níveis de qualidade da produção.

In [0]:
# agrupa os dados pela classificação de defeitos
df_manutencao = df_silver.groupBy("DefectStatus").agg(
    F.round(F.avg("MaintenanceHours"), 2).alias("HorasManutencaoMedia")
).orderBy("DefectStatus")

# converte o resultado para pandas
df_manutencao_pd = df_manutencao.toPandas()

# exibe a tabela
display(df_manutencao_pd)

In [0]:
# define os dados do gráfico
categorias = ["Baixo nível de defeitos", "Alto nível de defeitos"]
valores = df_manutencao_pd["HorasManutencaoMedia"]

# cria o gráfico
plt.figure(figsize=(9, 5))
plt.bar(categorias, valores)

# personaliza o gráfico
plt.title("Registros com alto nível de defeitos apresentam mais horas de manutenção")
plt.xlabel("Classificação de defeitos")
plt.ylabel("Média de horas de manutenção")

# exibe os valores acima das barras
for i, valor in enumerate(valores):
    plt.text(i, valor + 0.15, str(valor), ha="center")

plt.tight_layout()
plt.show()

### Interpretação do gráfico - Horas médias de manutenção
Apresenta uma diferença nas horas médias de manutenção entre os dois grupos. Os registros com alto nível de defeitos apresentam uma média de 12,37 horas de manutenção, enquanto os registros com baixo nível apresentam uma média de 6,79 horas.

Isso representa aproximadamente 82% mais horas de manutenção no grupo com alto nível de defeitos. Esse resultado indica uma associação entre maiores níveis de defeitos e maior necessidade de manutenção nos registros analisados. Entretanto, não é possível afirmar que o aumento das horas de manutenção causa o aumento dos defeitos, pois a análise é descritiva.

##7. Matriz de Correlação de Pearson
Considerando a pergunta de negócio: 
"Quais condições operacionais estão associadas a maiores níveis de defeitos na manufatura e como se relacionam com custos e eficiência?"

Primeiro, é preciso  investigar variáveis que possuem relações entre qualidade, manutenção, produção, custos e eficiência.

In [0]:
# faz uma lista daz 10 colunas selecionadas
variaveis_correlacao = [
    "DefectRate",
    "QualityScore",
    "MaintenanceHours",
    "DowntimePercentage",
    "ProductionVolume",
    "ProductionCost",
    "UnitCost",
    "EnergyConsumption",
    "EnergyEfficiency",
    "EnergyPerUnit"
]
# dataframe para utilizar na análise de Pearson
df_correlacao = df_silver.select(*variaveis_correlacao)
# visualiza os dados
display(df_correlacao)

In [0]:
# converte o dataframe PySpark para Pandas
df_correlacao_pd = df_correlacao.toPandas()

# calcula a matriz de correlação de Pearson
matriz_correlacao = df_correlacao_pd.corr(method="pearson")

# exibe a matriz
display(matriz_correlacao)

plt.figure(figsize=(12, 9))
plt.imshow(
    matriz_correlacao,
    cmap="coolwarm",
    vmin=-1,
    vmax=1
)

plt.colorbar(label="Correlação de Pearson")

plt.xticks(
    range(len(matriz_correlacao.columns)),
    matriz_correlacao.columns,
    rotation=45,
    ha="right"
)

plt.yticks(
    range(len(matriz_correlacao.columns)),
    matriz_correlacao.columns
)

# exibe os valores numéricos dentro das células
for i in range(len(matriz_correlacao.columns)):
    for j in range(len(matriz_correlacao.columns)):
        valor = matriz_correlacao.iloc[i, j]
        plt.text(j, i, f"{valor:.2f}", ha="center", va="center", fontsize=8)

plt.title("Matriz de Correlação de Pearson")
plt.tight_layout()
plt.show()

### Interpretação da matriz de correlação de Pearson

Em relação à taxa de defeitos (DefectRate), não foram identificadas correlações fortes.

Porém, foram identificadas relações entre alguns indicadores de eficiência operacional, como as correlações negativas entre volume de produção e custo unitário (-0,726), assim como entre volume de produção e consumo de energia por unidade (-0,708). Esses resultados indicam que registros com maior volume de produção tendem a apresentar menores custos e consumo energético por unidade.

Também foi observada uma correlação positiva entre custo unitário e consumo de energia por unidade (0,724), sugerindo que registros com maior custo por unidade tendem a apresentar maior consumo energético por unidade.

Esses resultados permitem explorar também aspectos relacionados à eficiência operacional da manufatura, mas destacando que correlação não implica causalidade necessariamente.

## Frente 2 — Eficiência operacional

Análise da relação entre volume de produção, custos e indicadores de eficiência operacional.

In [0]:

# comando para criar uma nova figura (contêiner principal para elementos)
plt.figure(figsize=(9, 6))
# função para criar gráfico de dispersão
plt.scatter(
    df_correlacao_pd["ProductionVolume"],
    df_correlacao_pd["UnitCost"],
    alpha=0.5
)

plt.title("Relação entre volume de produção e custo unitário")
plt.xlabel("Volume de produção")
plt.ylabel("Custo unitário")

plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### Interpretação de Volume de Produção X Custo Unitário

Para volumes BAIXOS, o custo unitário apresenta valores mais elevados e os pontos estão bastante dispersos. Isso indica maior variação do custo unitário entre os registros com menor volume de produção.

Para volumes ALTOS, o custo unitário tende a diminuir e os pontos ficam mais concentrados. 
Isso é coerente com a correlação negativa de -0,726 na matriz de Pearson.

Como o custo unitário é calculado dividindo o custo total pelo volume produzido, essa tendência pode ser parcialmente explicada pela fórmula. Ainda não é possível afirmar que produzir mais gera eficiência ou redução real de custos.


In [0]:
# comando para criar uma nova figura (contêiner principal para elementos)
plt.figure(figsize=(9, 6))
# comando para criar gráfico de dispersão
plt.scatter(
    df_correlacao_pd["ProductionVolume"],
    df_correlacao_pd["ProductionCost"],
    alpha=0.5
)

plt.title("Relação entre volume de produção e custo total")
plt.xlabel("Volume de produção")
plt.ylabel("Custo total de produção")

plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### Interpretação de Volume de Produção X Custo Total

Reafirma o que tinha sido visto na matriz de Pearson, que possui uma correlação de apenas 0,030 entre volume de produção e custo total, praticamente nenhuma associação linear. 

Entretanto, o custo unitário diminui conforme o volume aumenta. Será que essa redução do custo unitário representa um ganho operacional ou é principalmente consequência da forma como calculamos esse indicador?

In [0]:

# cria uma cópia dos dados para adicionar uma coluna e classificar os volumes de produção
df_analise = df_correlacao_pd.copy()

# cria uma nova coluna "FaixaVolume" 
df_analise["FaixaVolume"] = pd.qcut( # divide os registros em grupos com quantidades semelhantes
    df_analise["ProductionVolume"], # coluna usada para fazer a divisão
    q=4, # quantidade de quartis
    labels=["Volume baixo", "Volume médio-baixo",
            "Volume médio-alto", "Volume alto"] # nome de cada grupo
)

# variavel que recebe o resultado de cima
resumo_volume = df_analise.groupby(
    "FaixaVolume", observed=False # considera os valores nulos
).agg( # calcula estatísticas para cada grupo
    QuantidadeRegistros=("ProductionVolume", "count"),
    VolumeMedio=("ProductionVolume", "mean"),
    CustoTotalMedio=("ProductionCost", "mean"),
    CustoUnitarioMedio=("UnitCost", "mean")
).round(2)

display(resumo_volume)

### Análise do volume de produção e dos custos

Os registros foram divididos em quatro grupos conforme o volume produzido. O custo total médio permaneceu estável entre as faixas, variando aproximadamente entre 12.258,06 e 12.541,94.

O custo unitário médio teve uma redução progressiva conforme o volume de produção aumentou, passando de 65,68 na faixa de menor volume para 14,11 na faixa de maior volume.

Esse comportamento indica que volumes maiores estão associados a menores custos por unidade produzida. É importante considerar que o custo unitário é calculado pela divisão do custo total pelo volume de produção. Dessa forma, a redução pode estar relacionada ao próprio cálculo, porque o custo total médio apresentou pouca variação entre os grupos.

Os resultados demonstram uma associação entre volume de produção e custo unitário, mas isoladamente não permitem concluir que existe um ganho real de eficiência operacional ou economia.

## Frente 3 — Custos e energia

Análise das relações entre custos de produção e consumo de energia, considerando indicadores totais e por unidade produzida.

In [0]:

plt.figure(figsize=(9, 6))

plt.scatter(
    df_correlacao_pd["ProductionVolume"],
    df_correlacao_pd["EnergyConsumption"],
    alpha=0.5
)

plt.title("Relação entre volume de produção e consumo total de energia")
plt.xlabel("Volume de produção")
plt.ylabel("Consumo total de energia")

plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### Interpretação de Volume de Produção X Consumo Total de Energia

O resultado é coerente com a matriz de correlação. A correlação entre ProductionVolume e EnergyConsumption foi de aproximadamente -0,010, praticamente zero.

Isso significa que não foram encontradas evidências de que o aumento do volume de produção esteja associado ao aumento ou à redução do consumo total de energia.

In [0]:

resumo_energia = df_analise.groupby(
    "FaixaVolume", observed=False
).agg(
    QuantidadeRegistros=("EnergyConsumption", "count"),
    VolumeMedio=("ProductionVolume", "mean"),
    ConsumoTotalMedio=("EnergyConsumption", "mean"),
    ConsumoPorUnidadeMedio=("EnergyPerUnit", "mean")
).round(2)

display(resumo_energia)


### Análise do Volume de Produção e Eficiência Energética

O consumo total médio de energia permaneceu estável entre os grupos, variando de 2.936,34 a 3.037,93.

O consumo médio de energia por unidade apresentou uma redução conforme o volume de produção aumentou, passando de 16,05 na faixa de menor volume para 3,32 na faixa de maior volume.
Esse comportamento indica que os registros com maior volume de produção apresentam menor consumo energético por unidade produzida.

Esse valor é obtido pela divisão do consumo total de energia pelo volume de produção. Dessa forma, a redução pode ser explicada parcialmente pela própria composição do cálculo, considerando a estabilidade no consumo total médio.

Os resultados mostram uma associação entre volume de produção e consumo energético por unidade, mas não permitem concluir que existe um ganho real de eficiência energética.

### Correlação entre Custos e energia

Será investigado se existe uma relação entre o custo de produção e o consumo energético da manufatura. Na matriz de Pearson, teve uma correlação de +0,724 entre UnitCost e EnergyPerUnit.

É uma correlação positiva forte: registros com maior custo unitário tendem a apresentar também maior consumo de energia por unidade.

In [0]:

plt.figure(figsize=(9, 6))

plt.scatter(
    df_correlacao_pd["UnitCost"],
    df_correlacao_pd["EnergyPerUnit"],
    alpha=0.5
)

plt.title("Relação entre custo unitário e consumo de energia por unidade")
plt.xlabel("Custo unitário")
plt.ylabel("Consumo de energia por unidade")

plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Interpretação de Custo Unitário X Consumo de Energia por Unidade

Quando os valores são baixos, custo unitário e consumo energético por unidade estão concentrados próximos de zero. 

Quando os valores são intermediários e altos, os pontos começam a se espalhar, indicando maior variação entre os registros.

Em resumo, mesmo tendo essa dispersão crescente, a tendência geral pode continuar sendo crescente, o que é coerente com o Pearson de +0,724.

In [0]:

plt.figure(figsize=(9, 6))

plt.scatter(
    df_correlacao_pd["ProductionCost"],
    df_correlacao_pd["EnergyConsumption"],
    alpha=0.5
)

plt.title("Relação entre custo total e consumo total de energia")
plt.xlabel("Custo total de produção")
plt.ylabel("Consumo total de energia")

plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Interpretação de Custo Total X Consumo Total de Energia

A análise da relação entre custo total de produção (ProductionCost) e consumo total de energia (EnergyConsumption) não revelou um padrão visual evidente, visto que os pontos apresentam uma distribuição dispersa.

Esse comportamento é confirmado pelo coeficiente de correlação de Pearson de -0,007, indicando uma correlação linear praticamente inexistente.

A análise anterior identificou uma correlação positiva entre custo unitário (UnitCost) e consumo de energia por unidade (EnergyPerUnit), de aproximadamente 0,724.

Não foi identificada uma associação linear relevante entre custo total e consumo total de energia. A correlação pode estar relacionada à forma como eles são calculados, não sendo suficiente para afirmar que maiores custos unitários estão diretamente associados a maiores gastos energéticos.